# 04. RNN과 시퀀스 모델링

## 학습 목표
- 시퀀스 데이터를 처리하는 RNN의 구조와 원리 이해
- RNN을 순수 PyTorch Tensor로 직접 구현
- Vanishing Gradient 문제를 시각화로 확인
- LSTM의 Gate 메커니즘 이해
- Character-level 텍스트 생성 구현

## 참고 자료
- [Andrej Karpathy - The Unreasonable Effectiveness of RNNs](http://karpathy.github.io/2015/05/21/rnn-effectiveness/)
- [Chris Olah - Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/)

---

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import string
import random
import urllib.request

## 1. 시퀀스 데이터와 RNN의 필요성

**시퀀스 데이터**: 순서가 중요한 데이터
- 텍스트: "나는 학생이다" (단어 순서가 의미를 결정)
- 시계열: 주가, 온도 변화
- 음성: 시간에 따른 음파 신호

**FC/CNN의 한계**:
- 고정 크기 입력만 처리 가능
- 순서 정보를 반영하지 못함
- 이전 상태를 기억하지 못함

**RNN의 핵심 아이디어**: **hidden state**에 이전 정보를 저장하고, 새 입력과 함께 업데이트하며 순차적으로 처리

$$h_t = \tanh(W_{xh} x_t + W_{hh} h_{t-1} + b_h)$$
$$y_t = W_{hy} h_t + b_y$$

In [ ]:
# 시퀀스 데이터 예시: 간단한 패턴 예측
# sin 함수의 다음 값을 예측하는 문제

t = np.linspace(0, 4 * np.pi, 200)
signal = np.sin(t)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# 전체 신호
ax = axes[0]
ax.plot(t, signal, 'b-', linewidth=2)
ax.set_title('Sequence Data: sin(t)')
ax.set_xlabel('Time'); ax.set_ylabel('Value')
ax.grid(True, alpha=0.3)

# RNN의 처리 방식: 시간 단계별로 처리
ax = axes[1]
window = 20
ax.plot(t[:50], signal[:50], 'b-', linewidth=1, alpha=0.3)
# 현재 윈도우
ax.plot(t[10:10+window], signal[10:10+window], 'b-', linewidth=3, label='Input sequence')
ax.scatter([t[10+window]], [signal[10+window]], c='red', s=100, zorder=5, label='Predict next')
ax.axvline(x=t[10], color='gray', linestyle='--', alpha=0.5)
ax.axvline(x=t[10+window-1], color='gray', linestyle='--', alpha=0.5)
ax.set_title('RNN: Use past to predict future')
ax.set_xlabel('Time'); ax.set_ylabel('Value')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. RNN 구조: Hidden State와 시간 축 전개

### 접힌(folded) 형태
```
     ┌────┐
x_t ─┤ RNN ├── y_t
     └──┬─┘
        │ h_t (hidden state가 자기 자신으로 순환)
        └─────┘
```

### 펼친(unrolled) 형태
```
x_0      x_1      x_2      x_3
 │        │        │        │
 v        v        v        v
┌───┐    ┌───┐    ┌───┐    ┌───┐
│RNN│─h0─│RNN│─h1─│RNN│─h2─│RNN│─h3
└───┘    └───┘    └───┘    └───┘
 │        │        │        │
 v        v        v        v
y_0      y_1      y_2      y_3
```

**모든 시간 단계에서 동일한 가중치(W_xh, W_hh, W_hy)를 공유** -- 파라미터 공유

In [ ]:
# RNN 시간 축 전개 시각화

fig, ax = plt.subplots(figsize=(14, 5))

# RNN 블록 그리기
for t_step in range(5):
    x_pos = t_step * 3
    
    # RNN 박스
    rect = plt.Rectangle((x_pos - 0.5, 1.5), 1, 1, fill=True,
                          facecolor='lightblue', edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(x_pos, 2.0, 'RNN', ha='center', va='center', fontsize=11, fontweight='bold')
    
    # 입력 화살표 (아래에서 위로)
    ax.annotate('', xy=(x_pos, 1.5), xytext=(x_pos, 0.5),
                arrowprops=dict(arrowstyle='->', lw=2, color='green'))
    ax.text(x_pos, 0.2, f'$x_{{{t_step}}}$', ha='center', va='center', fontsize=13, color='green')
    
    # 출력 화살표 (위로)
    ax.annotate('', xy=(x_pos, 3.5), xytext=(x_pos, 2.5),
                arrowprops=dict(arrowstyle='->', lw=2, color='blue'))
    ax.text(x_pos, 3.8, f'$y_{{{t_step}}}$', ha='center', va='center', fontsize=13, color='blue')
    
    # Hidden state 화살표 (오른쪽으로)
    if t_step < 4:
        ax.annotate('', xy=(x_pos + 2.5, 2.0), xytext=(x_pos + 0.5, 2.0),
                    arrowprops=dict(arrowstyle='->', lw=2, color='red'))
        ax.text(x_pos + 1.5, 2.3, f'$h_{{{t_step}}}$', ha='center', va='center',
                fontsize=12, color='red')

# 초기 hidden state
ax.annotate('', xy=(-0.5, 2.0), xytext=(-2.0, 2.0),
            arrowprops=dict(arrowstyle='->', lw=2, color='red'))
ax.text(-2.5, 2.0, '$h_{-1}$\n(zeros)', ha='center', va='center', fontsize=11, color='red')

ax.set_xlim(-3.5, 14)
ax.set_ylim(-0.5, 4.5)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('RNN Unrolled Through Time', fontsize=14, pad=20)

plt.tight_layout()
plt.show()

---
## 3. RNN from Scratch

PyTorch의 `nn.RNN`을 사용하지 않고, 순수 Tensor 연산으로 RNN을 구현한다.

$$h_t = \tanh(x_t W_{xh} + h_{t-1} W_{hh} + b_h)$$
$$y_t = h_t W_{hy} + b_y$$

In [ ]:
class RNNFromScratch(nn.Module):
    """순수 Tensor 연산으로 구현한 RNN"""
    
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.hidden_size = hidden_size
        
        # 가중치 직접 정의 (nn.Linear 사용하지 않음)
        self.W_xh = nn.Parameter(torch.randn(input_size, hidden_size) * 0.01)
        self.W_hh = nn.Parameter(torch.randn(hidden_size, hidden_size) * 0.01)
        self.b_h = nn.Parameter(torch.zeros(hidden_size))
        
        self.W_hy = nn.Parameter(torch.randn(hidden_size, output_size) * 0.01)
        self.b_y = nn.Parameter(torch.zeros(output_size))
    
    def forward(self, x, h_prev=None):
        """
        x: (batch_size, seq_len, input_size)
        h_prev: (batch_size, hidden_size) or None
        
        Returns:
            outputs: (batch_size, seq_len, output_size)
            h_n: (batch_size, hidden_size)  # 최종 hidden state
        """
        batch_size, seq_len, _ = x.shape
        
        if h_prev is None:
            h_prev = torch.zeros(batch_size, self.hidden_size, device=x.device)
        
        outputs = []
        h_t = h_prev
        
        for t in range(seq_len):
            x_t = x[:, t, :]  # (batch_size, input_size)
            # RNN cell: h_t = tanh(x_t @ W_xh + h_{t-1} @ W_hh + b_h)
            h_t = torch.tanh(x_t @ self.W_xh + h_t @ self.W_hh + self.b_h)
            # 출력: y_t = h_t @ W_hy + b_y
            y_t = h_t @ self.W_hy + self.b_y
            outputs.append(y_t)
        
        outputs = torch.stack(outputs, dim=1)  # (batch_size, seq_len, output_size)
        return outputs, h_t

# 테스트
rnn_scratch = RNNFromScratch(input_size=10, hidden_size=20, output_size=5)
x_test = torch.randn(3, 7, 10)  # batch=3, seq_len=7, input=10
outputs, h_final = rnn_scratch(x_test)
print(f"입력 shape: {x_test.shape}")
print(f"출력 shape: {outputs.shape}  (batch, seq_len, output_size)")
print(f"최종 hidden shape: {h_final.shape}  (batch, hidden_size)")

In [ ]:
# PyTorch nn.RNN과 결과 비교
torch.manual_seed(42)

input_size, hidden_size = 4, 8
seq_len, batch_size = 5, 2

# 동일한 입력
x = torch.randn(batch_size, seq_len, input_size)

# 우리 구현
rnn_ours = RNNFromScratch(input_size, hidden_size, hidden_size)

# PyTorch RNN
rnn_pytorch = nn.RNN(input_size, hidden_size, batch_first=True)

# 가중치를 동일하게 설정
with torch.no_grad():
    rnn_ours.W_xh.copy_(rnn_pytorch.weight_ih_l0.T)
    rnn_ours.W_hh.copy_(rnn_pytorch.weight_hh_l0.T)
    rnn_ours.b_h.copy_(rnn_pytorch.bias_ih_l0 + rnn_pytorch.bias_hh_l0)
    rnn_ours.W_hy.copy_(torch.eye(hidden_size))  # identity
    rnn_ours.b_y.zero_()

# Forward
out_ours, h_ours = rnn_ours(x)
out_pt, h_pt = rnn_pytorch(x)

print("=== 출력 비교 ===")
print(f"출력 차이 (max): {(out_ours - out_pt).abs().max().item():.2e}")
print(f"Hidden 차이 (max): {(h_ours - h_pt.squeeze(0)).abs().max().item():.2e}")
print(f"-> 동일한 가중치를 사용하면 결과가 (거의) 일치!")

---
## 4. Vanishing Gradient 문제

RNN에서 역전파는 시간을 거슬러 올라간다 (Backpropagation Through Time, BPTT).

$$\frac{\partial L}{\partial h_0} = \frac{\partial L}{\partial h_T} \prod_{t=1}^{T} \frac{\partial h_t}{\partial h_{t-1}}$$

각 $\frac{\partial h_t}{\partial h_{t-1}}$에 $W_{hh}$가 곱해지므로:
- $\|W_{hh}\| < 1$이면: gradient가 **기하급수적으로 감소** (vanishing)
- $\|W_{hh}\| > 1$이면: gradient가 **기하급수적으로 증가** (exploding)

결과: **RNN은 긴 시퀀스에서 먼 과거의 정보를 학습하기 어렵다.**

In [ ]:
# Vanishing Gradient 시각화
# 시퀀스 길이에 따른 gradient 크기 변화

def measure_gradient_flow(seq_len, hidden_size=32):
    """주어진 시퀀스 길이에서 첫 번째 시간 단계의 gradient 크기 측정"""
    torch.manual_seed(42)
    rnn = nn.RNN(1, hidden_size, batch_first=True)
    
    # 입력 생성
    x = torch.randn(1, seq_len, 1, requires_grad=True)
    h0 = torch.zeros(1, 1, hidden_size)
    
    # Forward
    output, _ = rnn(x, h0)
    
    # 마지막 시간 단계의 출력에 대한 loss
    loss = output[0, -1, :].sum()
    loss.backward()
    
    # 각 시간 단계의 gradient 크기 (입력에 대한)
    grad = x.grad[0, :, 0].abs().detach().numpy()
    return grad

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, seq_len in zip(axes, [10, 30, 100]):
    grad = measure_gradient_flow(seq_len)
    ax.bar(range(seq_len), grad, color='steelblue', alpha=0.7)
    ax.set_xlabel('Time Step')
    ax.set_ylabel('|Gradient|')
    ax.set_title(f'Sequence Length = {seq_len}')
    ax.grid(True, alpha=0.3)

plt.suptitle('Vanishing Gradient: Gradient at early time steps vanishes for long sequences',
             fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("관찰: 시퀀스가 길어질수록 초기 시간 단계의 gradient가 0에 가까워진다.")
print("-> 먼 과거의 정보를 학습할 수 없음!")

In [ ]:
# RNN vs LSTM gradient 비교

def measure_gradient_rnn_vs_lstm(seq_len, hidden_size=32):
    results = {}
    for name, model_class in [('RNN', nn.RNN), ('LSTM', nn.LSTM)]:
        torch.manual_seed(42)
        model = model_class(1, hidden_size, batch_first=True)
        x = torch.randn(1, seq_len, 1, requires_grad=True)
        
        output, _ = model(x)
        loss = output[0, -1, :].sum()
        loss.backward()
        
        results[name] = x.grad[0, :, 0].abs().detach().numpy()
    return results

seq_len = 50
results = measure_gradient_rnn_vs_lstm(seq_len)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(seq_len), results['RNN'], 'r-', label='RNN', alpha=0.7, linewidth=2)
ax.plot(range(seq_len), results['LSTM'], 'b-', label='LSTM', alpha=0.7, linewidth=2)
ax.set_xlabel('Time Step')
ax.set_ylabel('|Gradient|')
ax.set_title('RNN vs LSTM: Gradient Flow Through Time')
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("LSTM은 gradient가 훨씬 잘 전파된다 -> 긴 시퀀스 학습 가능!")

---
## 5. LSTM (Long Short-Term Memory)

LSTM은 **cell state** $C_t$라는 별도의 경로를 도입하여 gradient가 잘 흐르게 한다.

### Gate 메커니즘

| Gate | 수식 | 역할 |
|------|------|------|
| Forget Gate | $f_t = \sigma(W_f [h_{t-1}, x_t] + b_f)$ | 이전 정보 중 **얼마나 잊을지** 결정 |
| Input Gate | $i_t = \sigma(W_i [h_{t-1}, x_t] + b_i)$ | 새 정보 중 **얼마나 저장할지** 결정 |
| Cell Update | $\tilde{C}_t = \tanh(W_C [h_{t-1}, x_t] + b_C)$ | 새로운 후보 정보 |
| Cell State | $C_t = f_t \odot C_{t-1} + i_t \odot \tilde{C}_t$ | 기억 업데이트 |
| Output Gate | $o_t = \sigma(W_o [h_{t-1}, x_t] + b_o)$ | **얼마나 출력할지** 결정 |
| Hidden State | $h_t = o_t \odot \tanh(C_t)$ | 최종 출력 |

**핵심**: Cell state $C_t$는 선형 경로 ($C_t = f_t \cdot C_{t-1} + ...$)를 통해 gradient가 잘 전파된다.

In [ ]:
# LSTM Gate 동작 시각화

# 간단한 예시: 괄호 안의 정보만 기억하기
# 입력: "A B ( C D ) E F" -> 괄호 안의 C, D만 기억

# LSTM gate 값을 시뮬레이션
tokens = ['A', 'B', '(', 'C', 'D', ')', 'E', 'F']
n_steps = len(tokens)

# 이상적인 gate 동작
forget_gate =   [1.0, 1.0, 0.0, 1.0, 1.0, 1.0, 0.0, 0.0]  # ( 에서 이전 기억 삭제
input_gate =    [0.0, 0.0, 0.0, 1.0, 1.0, 0.0, 0.0, 0.0]  # C, D만 저장
output_gate =   [0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 1.0, 1.0]  # ) 이후 출력

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)

gate_data = [
    ('Forget Gate $f_t$', forget_gate, 'tab:red',
     'f=0 at "(": forget previous memory'),
    ('Input Gate $i_t$', input_gate, 'tab:green',
     'i=1 at C,D: store new info'),
    ('Output Gate $o_t$', output_gate, 'tab:blue',
     'o=1 after ")": output stored info'),
]

for ax, (name, values, color, desc) in zip(axes, gate_data):
    bars = ax.bar(range(n_steps), values, color=color, alpha=0.7, edgecolor='black')
    ax.set_ylabel(name, fontsize=11)
    ax.set_ylim(0, 1.2)
    ax.set_xticks(range(n_steps))
    ax.set_xticklabels(tokens, fontsize=12, fontweight='bold')
    ax.text(0.02, 0.85, desc, transform=ax.transAxes, fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    ax.grid(True, alpha=0.3, axis='y')

axes[-1].set_xlabel('Input Tokens', fontsize=12)
plt.suptitle('LSTM Gate Activations: "Remember only what\'s inside parentheses"',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# LSTM 직접 구현 (한 셀)
class LSTMCell:
    """LSTM Cell 수동 구현 (교육용)"""
    
    def __init__(self, input_size, hidden_size):
        self.hidden_size = hidden_size
        # 4개의 gate를 하나의 행렬로 합침 (효율성)
        # [forget, input, cell_candidate, output] = 4 * hidden_size
        scale = np.sqrt(2.0 / (input_size + hidden_size))
        self.W = np.random.randn(input_size + hidden_size, 4 * hidden_size) * scale
        self.b = np.zeros(4 * hidden_size)
    
    def forward(self, x, h_prev, c_prev):
        H = self.hidden_size
        
        # 입력과 이전 hidden state 연결
        combined = np.concatenate([x, h_prev])
        
        # 한꺼번에 계산
        gates = combined @ self.W + self.b
        
        # 4개로 분리
        f_gate = self._sigmoid(gates[:H])          # Forget gate
        i_gate = self._sigmoid(gates[H:2*H])       # Input gate
        c_tilde = np.tanh(gates[2*H:3*H])          # Cell candidate
        o_gate = self._sigmoid(gates[3*H:4*H])     # Output gate
        
        # Cell state 업데이트
        c_t = f_gate * c_prev + i_gate * c_tilde
        
        # Hidden state
        h_t = o_gate * np.tanh(c_t)
        
        return h_t, c_t, (f_gate, i_gate, o_gate)
    
    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

# 테스트
lstm_cell = LSTMCell(input_size=3, hidden_size=4)
x = np.random.randn(3)
h = np.zeros(4)
c = np.zeros(4)

print("LSTM Cell 단계별 실행:")
for t in range(3):
    x_t = np.random.randn(3)
    h, c, (f, i, o) = lstm_cell.forward(x_t, h, c)
    print(f"  t={t}: h_norm={np.linalg.norm(h):.4f}, c_norm={np.linalg.norm(c):.4f}")
    print(f"         f_mean={f.mean():.3f}, i_mean={i.mean():.3f}, o_mean={o.mean():.3f}")

---
## 6. Character-level 텍스트 생성

LSTM으로 텍스트의 다음 글자를 예측하는 모델을 학습하고, 새로운 텍스트를 생성한다.

### 방법
1. 텍스트를 글자 단위로 분리
2. 입력: 글자 시퀀스, 출력: 다음 글자
3. LSTM 모델 학습
4. 시작 글자를 주고 한 글자씩 생성

In [ ]:
# 텍스트 데이터 준비
# Shakespeare의 작품 일부를 사용 (또는 간단한 텍스트)

# 작은 텍스트로 시작 (빠른 학습을 위해)
text = """To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles,
And by opposing end them. To die, to sleep;
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to, 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep: perchance to dream: ay, there's the rub;
For in that sleep of death what dreams may come
When we have shuffled off this mortal coil,
Must give us pause. There's the respect
That makes calamity of so long life;
For who would bear the whips and scorns of time,
The oppressor's wrong, the proud man's contumely,
The pangs of despised love, the law's delay,
The insolence of office and the spurns
That patient merit of the unworthy takes."""

# 문자 -> 숫자 매핑
chars = sorted(list(set(text)))
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}
vocab_size = len(chars)

print(f"텍스트 길이: {len(text)}")
print(f"고유 문자 수: {vocab_size}")
print(f"문자 목록: {''.join(chars)}")

# 텍스트를 인덱스로 변환
data = [char_to_idx[ch] for ch in text]
print(f"\n처음 50글자: '{text[:50]}'")
print(f"인덱스:       {data[:50]}")

In [ ]:
# 학습 데이터 생성
seq_length = 50  # 입력 시퀀스 길이

def create_sequences(data, seq_length):
    """연속된 문자 시퀀스에서 입력/타겟 쌍 생성"""
    X, Y = [], []
    for i in range(0, len(data) - seq_length):
        X.append(data[i:i+seq_length])
        Y.append(data[i+1:i+seq_length+1])  # 한 글자 뒤가 타겟
    return torch.tensor(X, dtype=torch.long), torch.tensor(Y, dtype=torch.long)

X_train, Y_train = create_sequences(data, seq_length)
print(f"학습 데이터: X shape={X_train.shape}, Y shape={Y_train.shape}")
print(f"\n예시:")
print(f"  입력:  '{''.join([idx_to_char[i.item()] for i in X_train[0]])}'")
print(f"  타겟:  '{''.join([idx_to_char[i.item()] for i in Y_train[0]])}'")

In [ ]:
# Character-level LSTM 모델
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x, hidden=None):
        # x: (batch, seq_len)
        embed = self.embedding(x)           # (batch, seq_len, embed_size)
        output, hidden = self.lstm(embed, hidden)  # (batch, seq_len, hidden_size)
        logits = self.fc(output)             # (batch, seq_len, vocab_size)
        return logits, hidden
    
    def generate(self, start_char, length=200, temperature=0.8):
        """텍스트 생성"""
        self.eval()
        chars_generated = [start_char]
        x = torch.tensor([[char_to_idx[start_char]]])
        hidden = None
        
        with torch.no_grad():
            for _ in range(length):
                logits, hidden = self(x, hidden)
                # Temperature scaling
                logits = logits[0, -1, :] / temperature
                probs = F.softmax(logits, dim=0)
                # 확률적 샘플링
                idx = torch.multinomial(probs, 1).item()
                chars_generated.append(idx_to_char[idx])
                x = torch.tensor([[idx]])
        
        return ''.join(chars_generated)

# 모델 생성
model = CharLSTM(
    vocab_size=vocab_size,
    embed_size=32,
    hidden_size=128,
    num_layers=2
)

total_params = sum(p.numel() for p in model.parameters())
print(f"모델 구조:\n{model}")
print(f"\n총 파라미터: {total_params:,}")

In [ ]:
# 학습 전 텍스트 생성 (랜덤)
print("=== 학습 전 생성 (랜덤) ===")
print(model.generate('T', length=100))
print()

In [ ]:
# 학습
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.003)

batch_size = 32
num_epochs = 200
losses = []

# DataLoader
dataset = torch.utils.data.TensorDataset(X_train, Y_train)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

model.train()
for epoch in range(num_epochs):
    epoch_loss = 0
    for batch_x, batch_y in dataloader:
        optimizer.zero_grad()
        logits, _ = model(batch_x)
        # (batch*seq_len, vocab_size) vs (batch*seq_len,)
        loss = criterion(logits.view(-1, vocab_size), batch_y.view(-1))
        loss.backward()
        # Gradient clipping (exploding gradient 방지)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        epoch_loss += loss.item()
    
    avg_loss = epoch_loss / len(dataloader)
    losses.append(avg_loss)
    
    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}")
        print(f"  Sample: {model.generate('T', length=80)}")
        print()
        model.train()

In [ ]:
# 학습 곡선
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(losses)
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Character LSTM Training Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 다양한 temperature로 텍스트 생성
print("=== Temperature에 따른 생성 결과 ===")
print("(낮은 temp = 더 보수적, 높은 temp = 더 창의적)\n")

for temp in [0.3, 0.8, 1.2]:
    print(f"--- Temperature = {temp} ---")
    generated = model.generate('T', length=150, temperature=temp)
    print(generated)
    print()

---
## 연습 문제

아래 문제를 직접 풀어보세요.

### 연습 1: RNN으로 시퀀스 예측

sin(x) 함수의 다음 값을 예측하는 RNN을 구현하세요.

In [ ]:
# TODO: sin(x) 다음 값 예측 RNN
#
# 1. 데이터 준비:
#    t = np.linspace(0, 8*np.pi, 400)
#    signal = np.sin(t)
#    window_size = 20으로 입력 시퀀스 생성
#    입력: signal[i:i+20], 타겟: signal[i+20]
#
# 2. 모델: nn.LSTM(1, 32, batch_first=True) + nn.Linear(32, 1)
#
# 3. 학습: MSELoss, Adam(lr=0.01), 100 epochs
#
# 4. 시각화: 실제 sin(x) vs 모델 예측


### 연습 2: 한국어 텍스트 생성

CharLSTM 모델을 한국어 텍스트에 적용해보세요.

In [ ]:
# TODO: 한국어 character-level 텍스트 생성
#
# 1. 한국어 텍스트 준비 (예: 소설 일부, 가사 등)
#    text_kr = """여기에 한국어 텍스트를 넣으세요.
#    긴 텍스트일수록 더 좋은 결과가 나옵니다."""
#
# 2. 위의 CharLSTM 모델을 그대로 사용 (vocab_size만 변경)
#
# 3. 학습 후 한국어 텍스트 생성
#
# 참고: 한국어는 유니크 문자 수가 많아 vocab_size가 커질 수 있음
#       embed_size와 hidden_size를 키워야 할 수 있음


---
## 핵심 정리

| 개념 | 핵심 내용 |
|------|----------|
| RNN | Hidden state로 시퀀스의 이전 정보를 기억. $h_t = \tanh(W_{xh}x_t + W_{hh}h_{t-1})$ |
| Unrolling | RNN을 시간 축으로 펼쳐서 보면 깊은 네트워크와 동일 |
| Vanishing Gradient | 긴 시퀀스에서 초기 gradient가 소멸 -> 장기 의존성 학습 불가 |
| LSTM | Cell state + 3개의 Gate로 gradient 흐름 개선 |
| Forget Gate | 이전 정보를 얼마나 잊을지 결정 |
| Input Gate | 새 정보를 얼마나 저장할지 결정 |
| Output Gate | 현재 상태를 얼마나 출력할지 결정 |
| Temperature | 텍스트 생성 시 다양성 조절. 낮으면 보수적, 높으면 창의적 |

**다음 단계**: Attention 메커니즘과 Transformer로 발전 (Transformer가 RNN을 대체한 이유를 배우게 됩니다)